# Next-stop delay model training (Trino + scikit-learn + MLflow)

Non-Spark alternative to `src/spark/jobs/ml/next_stop/train_delay_models.py`.
Reads the gold feature table through **Trino** into pandas, trains the same set of
classifiers, then logs and registers them to **MLflow** (champion alias on the best
ROC AUC).

Requirements: the kernel must have `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` set
so MLflow can push the model artifact to MinIO (S3). `MLFLOW_TRACKING_URI` and
`MLFLOW_S3_ENDPOINT_URL` are already provided by the image.

In [ ]:
%pip install --quiet trino

In [ ]:
import os

import mlflow
import mlflow.sklearn
import pandas as pd
import trino
from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

TRINO_HOST = os.getenv("TRINO_HOST", "trino.data-platform.svc.cluster.local")
TRINO_PORT = int(os.getenv("TRINO_PORT", "8080"))
TRINO_USER = os.getenv("TRINO_USER", "ml-training")
TRINO_CATALOG = os.getenv("TRINO_CATALOG", "dev")
GOLD_SCHEMA = os.getenv("GOLD_SCHEMA", "gold")
FEATURES_TABLE = os.getenv("FEATURES_TABLE", f"{TRINO_CATALOG}.{GOLD_SCHEMA}.next_stop_features")

EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT", "next_stop_delay")
REGISTERED_MODEL = os.getenv("MLFLOW_REGISTERED_MODEL", "next_stop_delay_classifier")
TRAIN_FRACTION = float(os.getenv("TRAIN_FRACTION", "0.8"))

LABEL_COL = "is_delayed"
CATEGORICAL_COLS = ["line_ref", "published_line", "direction", "cat_jour"]
NUMERIC_COLS = [
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "is_school_holiday",
    "hist_avg_arrival_delay_sec",
    "hist_pct_on_time",
]

In [ ]:
cols = CATEGORICAL_COLS + NUMERIC_COLS + [LABEL_COL, "service_date"]
query = (
    f"SELECT {', '.join(cols)} FROM {FEATURES_TABLE} "
    "WHERE is_delayed IS NOT NULL AND service_date IS NOT NULL"
)

conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user=TRINO_USER,
    catalog=TRINO_CATALOG,
    schema=GOLD_SCHEMA,
    http_scheme="http",
)
df = pd.read_sql(query, conn)
print("rows:", len(df))
df.head()

In [ ]:
df = df.sort_values("service_date")
df[NUMERIC_COLS] = df[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0)
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype(str).fillna("NA")

y = df[LABEL_COL].astype(int)
X = df[CATEGORICAL_COLS + NUMERIC_COLS]

cutoff = int(len(df) * TRAIN_FRACTION)
X_train, X_test = X.iloc[:cutoff], X.iloc[cutoff:]
y_train, y_test = y.iloc[:cutoff], y.iloc[cutoff:]
print("train:", len(X_train), "test:", len(X_test))

In [ ]:
preprocessor = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
        ("num", StandardScaler(), NUMERIC_COLS),
    ]
)

models = [
    ("logreg", LogisticRegression(max_iter=1000), {"model": "logistic_regression", "max_iter": 1000}),
    ("dtree", DecisionTreeClassifier(max_depth=8), {"model": "decision_tree", "max_depth": 8}),
    ("rforest", RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1), {"model": "random_forest", "n_estimators": 200, "max_depth": 10}),
    ("gbt", GradientBoostingClassifier(n_estimators=100, max_depth=5), {"model": "gradient_boosting", "n_estimators": 100, "max_depth": 5}),
]

In [ ]:
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()

results = []
for name, estimator, params in models:
    with mlflow.start_run(run_name=name) as run:
        pipe = Pipeline([("prep", preprocessor), ("clf", estimator)])
        pipe.fit(X_train, y_train)

        proba = pipe.predict_proba(X_test)[:, 1]
        preds = pipe.predict(X_test)
        roc_auc = roc_auc_score(y_test, proba)
        f1 = f1_score(y_test, preds)

        mlflow.set_tag("algorithm", name)
        mlflow.log_params(params)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("f1", f1)
        mlflow.sklearn.log_model(pipe, artifact_path="model")

        version = mlflow.register_model(f"runs:/{run.info.run_id}/model", REGISTERED_MODEL).version
        client.set_model_version_tag(REGISTERED_MODEL, version, "algorithm", name)
        client.set_model_version_tag(REGISTERED_MODEL, version, "roc_auc", f"{roc_auc:.4f}")

        print(f"{name}: roc_auc={roc_auc:.4f} f1={f1:.4f} version={version}")
        results.append((name, roc_auc, version))

In [ ]:
best_name, best_auc, best_version = max(results, key=lambda r: r[1])
client.set_model_version_tag(REGISTERED_MODEL, best_version, "best", "true")
client.set_registered_model_alias(REGISTERED_MODEL, "champion", best_version)
print(f"champion: {best_name} roc_auc={best_auc:.4f} version={best_version} -> alias 'champion'")